![](flopylogo_sm.png)

# Introduction to FloPy

The remainder of this workshop will use [FloPy](https://github.com/modflowpy/flopy), which is a Python package for creating, running, and post-processing MODFLOW-based groundwater flow and transport models. Why would we want to use this? MODFLOW, especially older versions, has idiosyncratic input and output that can be difficult to work with directly. FloPy translates MODFLOW input and output into the general Python data structures we explored in the first part of the course, making it easier to script groundwater modeling workflows with the entire scientific Python ecosysem.

 FloPy was originally developed by [Mark Bakker and others (2016)](https://ngwa.onlinelibrary.wiley.com/doi/10.1111/gwat.12413) for working with MODFLOW 2005 (including MODFLOW-NWT) and earlier versions of MODFLOW. [FloPy has since been expanded](https://doi.org/10.1111/gwat.13327) to support [MODFLOW 6](https://github.com/MODFLOW-USGS/modflow6), the current version of MODFLOW that includes general support for structured and unstructured grids, tight coupling of multiple models within a simulation, and a redesigned input structure that is meant to be more intuitive and human readable. FloPy support for MODFLOW 6 is tightly coupled to the MODFLOW 6 code, in that the relevant FloPy code is auto-generated from text files (_definition files_) that describe MODFLOW 6 models and packages. As a result, MODFLOW 2005-style and MODFLOW 6 functionality within FloPy is accessed through different subpackages—`flopy.modflow` and `flopy.mf6` respectively. 

For example, to instantiate a MODFLOW-NWT model instance, one would enter:


In [ ]:
# import some useful Python packages
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
import flopy

model = flopy.modflow.Modflow(modelname='test', version='mfnwt')  # or 'mf2005' for MODFLOW 2005 (the default)

For MODFLOW 6, we first have to create a simulation, which we then assign the model to:

In [3]:
sim = flopy.mf6.MFSimulation(sim_name='ps2b')
gwf = flopy.mf6.ModflowGwf(sim, modelname='ps2b')

As you can see, in addition to the different FloPy subpackages, the syntax is different between the two versions. In general, the syntax for the `flopy.modflow` and `flopy.mf6` subpackages follows the respective MODFLOW versions they support. For example, `ModflowGwf` is simply a [CamelCase](https://en.wikipedia.org/wiki/Camel_case) representation of the MODFLOW 6 Groundwater Flow (GWF) model. Similarly, the Discretization Package (for the GWF model) class is named `ModflowGwfdis`. Arguments to the `ModflowGwfdis` constructor follow the input variables to MODFLOW 6.

In [4]:
dis = flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=3,
    nrow=21,
    ncol=20,
    delr=500.,
    delc=500.,
    top=400.0,
    botm=[220, 200, 0],
)

The [MODFLOW 6 Input and Output Guide](https://modflow6.readthedocs.io/en/latest/mf6io.html) ([individual releases](https://github.com/MODFLOW-USGS/modflow6/releases) also contain a PDF version) can therefore be a valuable resource for understanding how to use Flopy.

Other Flopy subpackages (`flopy.modpath`, `flopy.mt3d`, etc) provide varying levels of support for previous versions of MODFLOW or related software such as MODPATH and MT3D; or general (Flopy-wide) support for discretization, exporting or other processing (`flopy.discretization`, `flopy.export`, `flopy.utils`, etc.).

This class will focus almost exclusively on FloPy support for MODFLOW 6. Examples and documentation for using FloPy in other contexts is available in the [FloPy online documentation](https://flopy.readthedocs.io/en/stable/index.html). The [Examples gallery](https://flopy.readthedocs.io/en/stable/examples.html) are a resource that can be used to quickly learn the underlying capabilities of FloPy.

In [5]:
mg = gwf.modelgrid

### Building a first model with FloPy

In this activity the class will build a version of one of the primary teaching models from the "Introduction to MODFLOW class". The model, named problem set 2, is a three layer MODFLOW model with recharge, stream, and sometimes pumping boundary conditons. The figure below shows the basic layout of the model.

![](PS2b_example.png)

For this model, we'll be constructing it with 500x500 ft grid cells

#### Start by creating a simulation object (`flopy.mf6.MFSimulation()`)

We'll set the simulation path for this model (the location where files will be written) to `"./ps2b"`. FloPy will create the directory for us when we write the simulation to disk.

In [ ]:
sim_path = Path("./ps2b")

#### Now add an solver object and time discretization to the simulation object

#### Generate a Groundwater Flow model object to begin building the model (`flopy.mf6.ModflowGwf`)

#### Build a Discretization object (Modflow DIS package) using FloPy.
For this model the top elevation will be set to 350 ft. Grid cell size is 500 x 500 ft.

#### Build a Node Property Flow object (MODFLOW NPF package)

The node property flow package is used to set hydraulic parameters (e.g., horizontal hydraulic conductivity) for the MODFLOW model

#### Build an Initial conditions package for MODFLOW (MODFLOW IC package)

The initial conditions package is used to set the starting heads for a model. This information is needed by MODFLOW to simulate groundwater flow.

The starting heads for this model are 330 ft

#### Add the river boundary condition

The river boundary condition could be represented multiple ways in MODFLOW. Options include the Constant head package (CHD), General head package (GHB), Drain package (DRN), or the River package (RIV). Each of these representations has their own unique advantages and limitations. For this example, we'll be using the RIV package to simulate the river boundary condition.

*Information we need to simulate the river boundary condition:*
   - The river bed has a hydraulic conductivity of 20 ft/d and is 1 ft thick
   - The river is 10 ft wide
   - The elevation of the river bottom is 317 ft.
   - The stage in the river is 320 ft.

#### Add the recharge boundary condition

In this example we'll use the `flopy.mf6.ModflowGwfrcha` package, which allows us to specify recharge in an array format

#### Set up output using the output control package

We're going to save outputs to a binary head file and a CSV file of budget information.

#### Write the model to file and run the simulation

#### Look at head outputs

In [ ]:
# plot up heads


In [ ]:
## plot up a cross-section


#### Look at budget outputs

#### IF TIME PERMITS: Change the model by adding in a well to simulate pumping

For this example we're going to add in a well to simulate pumping in layer 1. The well location will be near the center of the model (0, 9, 10) and have a pumping rate of 75,000 ft / day.

**Our first step is to change the simulation path so we don't overwrite our existing model**

In [ ]:
new_path = Path("./ps2d")

#### Add in the well package using `flopy.mf6.ModflowGwfwel`

Write and run the new simulation

And now let's load the budget and compare components to the original simulation